In [ ]:
!pip install ipykernel jupyter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Nạp dữ liệu từ file tương ứng
# Dựa trên file .names, chúng ta có 9 cột (8 đặc trưng + 1 nhãn Outcome)
column_names = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

# Đọc file dữ liệu gốc
df = pd.read_csv('pima-indians-diabetes.data.csv', names=column_names)

# Hiển thị 5 dòng đầu để kiểm tra
print("Dữ liệu 5 dòng đầu tiên:")
display(df.head())

# Cấu hình chung cho biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8) 

: 

### Biểu đồ 1: Tỷ lệ người mắc bệnh tiểu đường

Biểu đồ thanh này thể hiện số lượng người được chẩn đoán mắc bệnh tiểu đường (Outcome = 1) và không mắc bệnh (Outcome = 0) trong tập dữ liệu. Nó giúp chúng ta có cái nhìn tổng quan về sự mất cân bằng giữa hai lớp.

In [ ]:
# ---------------------------------------------------------
# BIỂU ĐỒ 1: Tỷ lệ người mắc bệnh tiểu đường
# ---------------------------------------------------------
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Outcome', data=df, palette='Set1')
plt.title('Tỷ lệ mắc bệnh tiểu đường (0: Không, 1: Có)', fontsize=14)
plt.xlabel('Kết quả (Outcome)')
plt.ylabel('Số lượng mẫu')
# Thêm nhãn số lượng trên đầu cột
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                textcoords='offset points')
plt.show()

Biểu đồ này cho thấy có khoảng 500 người không mắc tiểu đường và hơn 250 người mắc tiểu đường trong tập dữ liệu.

### Biểu đồ 2: Ma trận tương quan (Heatmap)

Biểu đồ Heatmap hiển thị mối quan hệ tương quan giữa các đặc trưng khác nhau trong tập dữ liệu. Các giá trị gần 1 hoặc -1 cho thấy mối tương quan mạnh mẽ (tích cực hoặc tiêu cực), trong khi giá trị gần 0 cho thấy ít hoặc không có tương quan. Biểu đồ này giúp xác định các đặc trưng có thể ảnh hưởng đến kết quả `Outcome`.

In [ ]:
# ---------------------------------------------------------
# BIỂU ĐỒ 2: Ma trận tương quan (Heatmap)
# ---------------------------------------------------------
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdYlGn', fmt='.2f', linewidths=0.5)
plt.title('Ma trận tương quan giữa các yếu tố sức khỏe', fontsize=15)
plt.show()

Từ ma trận tương quan, chúng ta có thể thấy `Glucose` có mối tương quan mạnh nhất với `Outcome`, tiếp theo là `BMI` và `Age`.

### Biểu đồ 3: Phân bổ các chỉ số quan trọng (Glucose, BMI, Age, BloodPressure)

Các biểu đồ phân phối (histogram với KDE) cho từng chỉ số quan trọng như `Glucose`, `BMI`, `Age` và `BloodPressure`, được chia theo nhóm `Outcome`. Điều này giúp chúng ta hiểu cách phân bổ của các chỉ số này khác nhau giữa những người mắc bệnh và không mắc bệnh tiểu đường.

In [ ]:
# ---------------------------------------------------------
# BIỂU ĐỒ 3: Phân bổ các chỉ số quan trọng (Glucose, BMI, Age, BloodPressure)
# ---------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Danh sách các cột quan trọng nhất để vẽ
important_features = ['Glucose', 'BMI', 'Age', 'BloodPressure']

for i, col in enumerate(important_features):
    sns.histplot(data=df, x=col, hue='Outcome', kde=True, ax=axes[i//2, i%2], palette='magma', element="step")
    axes[i//2, i%2].set_title(f'Phân bổ chỉ số {col} theo Nhóm Bệnh', fontsize=12)

plt.tight_layout()
plt.show()

Các biểu đồ phân phối cho thấy rõ ràng những người mắc tiểu đường (Outcome=1) thường có giá trị `Glucose`, `BMI` và `Age` cao hơn so với những người không mắc bệnh (Outcome=0).

### Biểu đồ 4: Boxplot để xem giá trị ngoại lai và khoảng dữ liệu (Dữ liệu đã chuẩn hóa)

Biểu đồ Boxplot này được tạo sau khi chuẩn hóa các đặc trưng để chúng có thể được so sánh trên cùng một thang đo. Biểu đồ này giúp chúng ta hình dung sự phân bố của từng đặc trưng và xác định các giá trị ngoại lai, cũng như sự khác biệt về phân phối giữa hai nhóm `Outcome`.

In [ ]:
# ---------------------------------------------------------
# BIỂU ĐỒ 4: Boxplot để xem giá trị ngoại lai và khoảng dữ liệu
# ---------------------------------------------------------
plt.figure(figsize=(15, 6))
df_melted = df.drop('Outcome', axis=1)
# Chuẩn hóa dữ liệu một chút để vẽ chung (vì Insulin và Pedigree lệch nhau quá nhiều)
df_norm = (df_melted - df_melted.mean()) / df_melted.std()
df_norm['Outcome'] = df['Outcome']
df_plot = pd.melt(df_norm, id_vars='Outcome', var_name='Features', value_name='Standardized Value')

sns.boxplot(x='Features', y='Standardized Value', hue='Outcome', data=df_plot, palette='Set2')
plt.xticks(rotation=45)
plt.title('So sánh các chỉ số giữa 2 nhóm (Dữ liệu đã chuẩn hóa)', fontsize=15)
plt.show()

Boxplot cho thấy sự khác biệt trong phân bố của các đặc trưng giữa hai nhóm, đặc biệt là `Glucose`, `BMI` và `Age`, ngay cả sau khi chuẩn hóa. Điều này làm nổi bật tầm quan trọng của các đặc trưng này trong việc phân loại bệnh tiểu đường.